# 13장. 외부 데이터로 분석을 확장하기

이 노트북은 외부 데이터를 무조건 수집하는 실습이 아닙니다. 공식 출처를 우선하고, 인증 정보와 정책을 확인하고, 원본·정제 결과·메타데이터를 분리하는 안전한 수집 흐름을 연습합니다.

네트워크 호출은 기본적으로 비활성화되어 있습니다.


## 학습 목표

- 공식 파일, 공식 API, 제한적 크롤링의 우선순위를 구분합니다.
- API Key를 코드와 로그에서 분리합니다.
- timeout, 오류 처리, 제한적 재시도가 포함된 HTTP 요청을 사용합니다.
- robots.txt와 이용약관을 각각 확인합니다.
- 원본·정제 결과·출처 메타데이터·파일 해시를 분리해 저장합니다.
- 외부 데이터 병합 후 행 수와 미매칭을 검증합니다.


## 1. 프로젝트 루트 설정


In [ ]:
from pathlib import Path
import sys


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / 'requirements.txt').exists()
            and (candidate / 'scripts').exists()
        ):
            return candidate
    raise FileNotFoundError(
        '프로젝트 루트 폴더를 찾을 수 없습니다.'
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('프로젝트 루트:', PROJECT_ROOT)
print('보고서 폴더:', REPORT_DIR)


## 2. 네트워크 호출 없이 준비 자료 생성

먼저 폴더, 수집 계획, 연결 기준, 체크리스트, 출처 로그 템플릿을 만듭니다.


In [ ]:
from src.external_data_collection import (
    run_external_data_collection_setup,
)

setup_result = run_external_data_collection_setup(
    base_dir=PROJECT_ROOT,
    report_dir=REPORT_DIR,
)

setup_result['paths']


In [ ]:
outputs = setup_result['outputs']

display(outputs['data_plan'])
display(outputs['method_summary'])
display(outputs['integration_plan'])
display(outputs['env_status'])


## 3. 인증 정보 확인

실제 Key 값은 출력하지 않고 로드 여부만 확인합니다. `.env.example`을 `.env`로 복사한 뒤 개인 발급값을 입력합니다.


In [ ]:
outputs['env_status']


## 4. 안전한 HTTP Session 준비

`429`와 일부 일시적 서버 오류에만 제한적으로 재시도합니다. 연결 timeout과 읽기 timeout은 공통 모듈에 설정되어 있습니다.


In [ ]:
from src.external_data_collection import build_http_session

session = build_http_session(
    total_retries=3,
    backoff_factor=0.5,
)


## 5. 네이버 검색 API 선택 실행

공식 문서와 호출 한도를 확인하고, `.env`에 인증 정보가 준비된 경우에만 `RUN_NAVER_API=True`로 변경합니다.


In [ ]:
from src.external_data_collection import (
    naver_blog_items_to_dataframe,
    save_json_snapshot,
    search_naver_blog,
    sha256_file,
)

RUN_NAVER_API = False

if RUN_NAVER_API:
    result, metadata = search_naver_blog(
        '제주 여행',
        display=10,
        start=1,
        sort='sim',
        session=session,
    )

    raw_path = save_json_snapshot(
        result,
        setup_result['paths']['raw']
        / 'naver_blog_jeju.json',
    )

    naver_blog_df = naver_blog_items_to_dataframe(
        result
    )
    processed_path = (
        setup_result['paths']['processed']
        / 'naver_blog_jeju.csv'
    )
    naver_blog_df.to_csv(
        processed_path,
        index=False,
        encoding='utf-8-sig',
    )

    metadata['raw_sha256'] = sha256_file(raw_path)

    display(metadata)
    display(naver_blog_df.head())
else:
    print(
        '네트워크 호출을 실행하지 않았습니다. '
        '공식 문서와 인증 정보를 확인한 뒤 '
        'RUN_NAVER_API=True로 변경하세요.'
    )


## 6. 크롤링 정책 확인 후 선택 실행

`robots.txt`는 자동 접근 규칙이며 이용허락 자체가 아닙니다. 이용약관, 라이선스, 개인정보, 콘텐츠 사용 범위를 별도로 확인하고 `POLICY_CONFIRMED=True`로 설정합니다.


In [ ]:
from src.external_data_collection import (
    extract_title_and_links,
    fetch_public_html,
    save_text_snapshot,
)

RUN_CRAWLING_EXAMPLE = False
POLICY_CONFIRMED = False
TARGET_URL = 'https://example.com/'

if RUN_CRAWLING_EXAMPLE and POLICY_CONFIRMED:
    html, metadata = fetch_public_html(
        TARGET_URL,
        policy_confirmed=True,
        respect_robots=True,
        session=session,
    )

    raw_html_path = save_text_snapshot(
        html,
        setup_result['paths']['raw']
        / 'example_page.html',
    )

    page_title, links_df = extract_title_and_links(
        html,
        base_url=TARGET_URL,
    )
    links_df.to_csv(
        setup_result['paths']['processed']
        / 'example_links.csv',
        index=False,
        encoding='utf-8-sig',
    )

    print('페이지 제목:', page_title)
    display(metadata)
    display(links_df.head())
else:
    print(
        '크롤링을 실행하지 않았습니다. '
        '이용약관과 robots.txt를 각각 확인하세요.'
    )


## 7. 외부 데이터 병합 검증

외부 데이터는 날짜, 지역, 카테고리 등의 단위를 맞춘 뒤 병합합니다. 아래 예시는 행 수와 미매칭을 확인하는 구조입니다.


In [ ]:
import pandas as pd

from src.external_data_collection import (
    merge_external_data,
)

monthly_sales = pd.DataFrame({
    'order_month': ['2026-01', '2026-02', '2026-03'],
    'total_sales': [1200000, 1500000, 1300000],
})

external_monthly = pd.DataFrame({
    'order_month': ['2026-01', '2026-02', '2026-03'],
    'holiday_count': [3, 1, 2],
})

merged_monthly, merge_check = merge_external_data(
    monthly_sales,
    external_monthly,
    on='order_month',
    how='left',
    validate='many_to_one',
)

display(merged_monthly)
display(merge_check)


## 8. 체크리스트와 출처 로그 확인

실제 수집 후에는 체크리스트의 상태와 메모를 채우고, 제공 기관·URL·기준일·UTC 수집 시각·원본 파일 해시를 기록합니다.


In [ ]:
display(outputs['checklist'])
display(outputs['external_data_log'])


## 9. 생성된 결과 파일 확인


In [ ]:
for name, path in setup_result['output_paths'].items():
    print(
        name,
        'OK' if path.exists() else 'MISSING',
        path,
    )


## 10. 스크립트로 다시 실행하기

프로젝트 루트의 터미널에서 다음 명령으로 같은 준비 자료를 다시 생성할 수 있습니다.

```powershell
python scripts/run_external_data_collection.py
```


## 정리

외부 데이터 수집에서 중요한 것은 요청 코드를 빨리 작성하는 것이 아닙니다. 공식 출처와 정책을 확인하고, 필요한 범위만 요청하고, 인증 정보를 숨기고, 원본·정제 결과·메타데이터를 분리하고, 병합 결과를 검증하는 것이 핵심입니다.
